# 01 — Dataset Exploration

Exploratory data analysis (EDA) on the Roboflow Pothole Dataset v2.
Run `python training/dataset_prep.py` first to download the dataset.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt

DATASET = Path('../data/datasets/pothole-detection-v2')
data_yaml = DATASET / 'data.yaml'
cfg = yaml.safe_load(data_yaml.read_text()) if data_yaml.exists() else {}
cfg

In [ ]:
# Count images per split
splits = ['train', 'valid', 'test']
counts = {s: len(list((DATASET / s / 'images').glob('*'))) if (DATASET / s / 'images').exists() else 0 for s in splits}
pd.Series(counts).plot.bar(title='Images per split'); plt.show()
counts

In [ ]:
# Class distribution from YOLO label files
from collections import Counter
labels_dir = DATASET / 'train' / 'labels'
class_counter = Counter()
if labels_dir.exists():
    for f in labels_dir.glob('*.txt'):
        for line in f.read_text().splitlines():
            if line.strip():
                class_counter[int(line.split()[0])] += 1
names = cfg.get('names', {})
pd.Series({names.get(k, k): v for k, v in class_counter.items()}).plot.bar(title='Class distribution'); plt.show()
dict(class_counter)

In [ ]:
# Bounding box size distribution (normalised area)
areas = []
if labels_dir.exists():
    for f in labels_dir.glob('*.txt'):
        for line in f.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:
                _, _, _, w, h = map(float, parts)
                areas.append(w * h)
if areas:
    plt.hist(areas, bins=40); plt.title('Normalised bbox area'); plt.show()
len(areas)